# Semantic Embedding 
Here we perform a semantic embedding of all articles in our datasets using Google's Gemma3 [(Hugging Face)](https://huggingface.co/google/embeddinggemma-300m)

code by Brandon G. Villalta Lopez

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [2]:
df = pd.read_csv('Data/EDA_data-FULL.csv')
df.head()

,Apikey,URL,Title,text,Publish_date,Authors,Section,User_Needs,Views,Avg. views,Engaged minutes,Avg. minutes,Desktop views,Mobile views,Tablet views
0,"pressherald.com, sunjournal.com",http://www.pressherald.com/2026/01/09/gray-inv...,Gray investigated for buying $1.25M fire truck...,Federal and local law enforcement are investig...,2026-01-09 08:55:00,Rory Sweeting,News,update-me,109875,1.084,31960.0,0.315,3910.0,104307.0,1658.0
1,"centralmaine.com, pressherald.com, sunjournal.com",https://www.pressherald.com/2025/03/06/social-...,Social Security now requires Maine parents to ...,"An update, in which the Social Security Admini...",2025-03-06 17:14:00,Joe Lawlor,News,none,98329,1.109,64495.0,0.727,9783.0,86241.0,2305.0
2,"centralmaine.com, pressherald.com, sunjournal.com",http://www.pressherald.com/2026/01/28/ice-agen...,"ICE agents shatter window, leave 1-month-old b...","PORTLAND — Hassane Barry and his wife, Nene, b...",2026-01-28 17:43:00,"Dylan Tusinski,Salomé Cloteaux",News,give-me-perspective,76168,1.145,84939.0,1.277,12634.0,61819.0,1715.0
3,"centralmaine.com, pressherald.com, sunjournal.com",https://www.pressherald.com/2025/02/05/maine-m...,Maine Mall shooting: Police search for suspect...,SOUTH PORTLAND — Police are searching for a pe...,2025-02-05 16:21:00,"Daniel Kool,Morgan Womack",News,none,73901,1.341,38065.0,0.691,14806.0,58483.0,612.0
4,"centralmaine.com, pressherald.com, sunjournal.com",http://www.pressherald.com/2025/10/21/graham-p...,Graham Platner says he will remove a Nazi-link...,A leading Democratic candidate for U.S. Senate...,2025-10-21 12:27:00,Randy Billings,Politics,educate-me,62651,1.158,29502.0,0.545,16738.0,45454.0,459.0


In [3]:
# Download from the 🤗 Hub --> https://huggingface.co/google/embeddinggemma-300m
model = SentenceTransformer("google/embeddinggemma-300m")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

## clustering-optimized embedding

In [4]:
cluster_documents = ['title: {title | "'+title+'"} | task: clustering | text: {'+text+'}' for title, text in zip(df.Title,df.text)]

cluster_embeddings = model.encode_document(cluster_documents)
cluster_embeddings.shape

(8606, 768)

## classification-optimized embedding

In [5]:
class_documents = ['title: {title | "'+title+'"} | task: classification | text: {'+text+'}' for title, text in zip(df.Title,df.text)]

class_embeddings = model.encode_document(class_documents)
class_embeddings.shape

(8606, 768)

## merge and save

In [15]:
import h5py

with h5py.File("Data/semantic_embeddings.h5", "w") as f:
    f.create_dataset("cluster_embedding", data=cluster_embeddings)       
    f.create_dataset("classification_embedding", data=class_embeddings)       
    f.create_dataset("URL", data=df.URL.to_numpy().astype('S'))

In [16]:
print('Saved!')

Saved!
